# NESTML/SpiNNaker balanced network

In this notebook, we will create a recurrent spiking network using the newly created NESTML neuron model. 

In [1]:
%matplotlib inline

import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gs
import matplotlib.pyplot as pylab
import numpy
from pyNN.random import RandomDistribution
from pyNN.utility.plotting import Figure, Panel
import pyNN.spiNNaker as p
import pyNN.spiNNaker as pynn
from pyNN.utility.plotting import Figure, Panel
from spynnaker.pyNN.data import SpynnakerDataView

SpynnakerDataView.register_binary_search_path("/opt/app-root/src")

from iaf_psc_exp_neuron_nestml import iaf_psc_exp_neuron_nestml

In [2]:
use_nestml = True

if use_nestml:
    receptor_name_exc = "exc_spikes"
    receptor_name_inh = "inh_spikes"
    membranePot = "V_m"
else:
    receptor_name_exc = "excitatory"
    receptor_name_inh = "inhibitory"
    membranePot = "v"

In [3]:
neuron_parameters = {'C_m': 250,
                     'tau_m': 10,
                     'tau_syn_inh': 2,
                     'tau_syn_exc': 2,
                     'refr_T': 2,
                     'E_L': -70,
                     'V_reset': -70,
                     'V_th': -55}

if not use_nestml:
    # different names/scaling for the parameters
    neuron_parameters = {'tau_m': neuron_parameters["tau_m"],
                          'cm': 1E-3 * neuron_parameters["C_m"],
                          'v_rest': neuron_parameters["E_L"],
                          'v_reset': neuron_parameters["V_reset"],
                          'v_thresh': neuron_parameters["V_th"],
                          'tau_syn_E': neuron_parameters["tau_syn_exc"],
                          'tau_syn_I': neuron_parameters["tau_syn_inh"],
                          'tau_refrac': neuron_parameters["refr_T"]}

In [4]:
p.setup(timestep=1)

if use_nestml:
    p.set_number_of_neurons_per_core(p.IF_curr_exp, 4)
else:
    p.set_number_of_neurons_per_core(iaf_psc_exp_neuron_nestml(), 4)
    
p.set_number_of_neurons_per_core(p.SpikeSourcePoisson, 4)

t_sim = 1000    # total time to simulator for [ms]
p_conn = .1    # connection probability
rate_ext_input = 50.    # [s⁻¹]
n_neurons = 500
n_exc = int(round(n_neurons * 0.8))
n_inh = int(round(n_neurons * 0.2))

if use_nestml:
    weight_exc = 1E3 * 0.1
    weight_inh = -5.0 * weight_exc
    weight_input = 1E3 * 1.
else:
    weight_exc = 0.1
    weight_inh = -5.0 * weight_exc
    weight_input = 0.001

pop_input = p.Population(100, p.SpikeSourcePoisson(rate=0.0),
                         additional_parameters={
                             "max_rate": 50.0,
                             "seed": 0},
                         label="Input")

#
# excitatory and inhibitory populations
#

if use_nestml:
    pop_exc = p.Population(n_exc, iaf_psc_exp_neuron_nestml(), label="Excitatory", seed=1)
    pop_inh = p.Population(n_inh, iaf_psc_exp_neuron_nestml(), label="Inhibitory", seed=2)
else:
    pop_exc = p.Population(n_exc, p.IF_curr_exp, label="Excitatory", seed=1)
    pop_inh = p.Population(n_inh, p.IF_curr_exp, label="Inhibitory", seed=2)
    
pop_exc.set(**neuron_parameters)
pop_inh.set(**neuron_parameters)


#
# external stimulus to exc and inh populations
#

stim_exc = p.Population(
    n_exc, p.SpikeSourcePoisson(rate=50.0), label="Stim_Exc",
    additional_parameters={"seed": 3})
stim_inh = p.Population(
    n_inh, p.SpikeSourcePoisson(rate=50.0), label="Stim_Inh",
    additional_parameters={"seed": 4})

p.Projection(stim_exc, pop_exc, p.OneToOneConnector(), p.StaticSynapse(weight=weight_input, delay=1.), receptor_type=receptor_name_exc)
p.Projection(stim_inh, pop_inh, p.OneToOneConnector(), p.StaticSynapse(weight=weight_input, delay=1.), receptor_type=receptor_name_exc)


#
# recurrent connections
#

delays_exc = RandomDistribution("normal_clipped", mu=1.5, sigma=0.75, low=1.0, high=1.6)
weights_exc = RandomDistribution("normal_clipped", mu=weight_exc, sigma=0.1, low=0, high=numpy.inf)
conn_exc = p.FixedProbabilityConnector(p_conn)
synapse_exc = p.StaticSynapse(weight=weights_exc, delay=delays_exc)
delays_inh = RandomDistribution("normal_clipped", mu=0.75, sigma=0.375, low=1.0, high=1.6)
weights_inh = RandomDistribution("normal_clipped", mu=weight_inh, sigma=0.1, low=-numpy.inf, high=0)
conn_inh = p.FixedProbabilityConnector(p_conn)
synapse_inh = p.StaticSynapse(weight=weights_inh, delay=delays_inh)
p.Projection(pop_exc, pop_exc, conn_exc, synapse_exc, receptor_type=receptor_name_exc)
p.Projection(pop_exc, pop_inh, conn_exc, synapse_exc, receptor_type=receptor_name_exc)
p.Projection(pop_inh, pop_inh, conn_inh, synapse_inh, receptor_type=receptor_name_inh)
p.Projection(pop_inh, pop_exc, conn_inh, synapse_inh, receptor_type=receptor_name_inh)

if use_nestml:
    pop_exc.initialize(V_m=RandomDistribution("uniform", low=-65.0, high=-55.0))
    pop_inh.initialize(V_m=RandomDistribution("uniform", low=-65.0, high=-55.0))
else:
    pop_exc.initialize(v=RandomDistribution("uniform", low=-65.0, high=-55.0))
    pop_inh.initialize(v=RandomDistribution("uniform", low=-65.0, high=-55.0))

pop_exc[:5].record([membranePot])    # record only from the first 5 neurons
pop_inh.record("spikes")
pop_exc.record("spikes")

pop_input.set(rate=rate_ext_input)

p.run(t_sim)

pop_exc_spikes = pop_exc.get_data("spikes")
pop_inh_spikes = pop_inh.get_data("spikes")
v_neuron = pop_exc.get_data(membranePot)
end_time = p.get_current_time()

p.end()

2026-06-11 13:56:50 INFO: Read configs files: /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinn_utilities/spinn_utilities.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinn_machine/spinn_machine.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/pacman/pacman.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinnman/spinnman.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/var/spack/environments/ebrains-25-10/.spack-env/view/lib/python3.11/site-packages/spinn_front_end_common/interface/spinnaker.cfg, /srv/main-spack-instance-2502/ebrains-spack-builds/vendor/spack/

In [5]:
# plot membrane potential traces

fig, ax = plt.subplots(nrows=1, dpi=150)

ax.set_ylabel("Membrane potential [mV]")
ax.plot(v_neuron.segments[0].filter(name=membranePot)[0])

plt.savefig("spinnaker_balanced_network_V_m_[use_nestml="+str(use_nestml)+"].png")

In [6]:
# plot rastergram

all_exc_spikes = pop_exc_spikes.segments[0].spiketrains
all_inh_spikes = pop_inh_spikes.segments[0].spiketrains

fig, ax = plt.subplots()

for (nrn, spike_train) in enumerate(all_exc_spikes):
    ax.plot(spike_train, np.array(np.ones_like(spike_train) * nrn) + n_inh, "|", c='r', ms=5)

for (nrn, spike_train) in enumerate(all_inh_spikes):
    ax.plot(spike_train, np.ones_like(spike_train) * nrn, "|", c='b', ms=5)

fig.savefig("recurrent_spinnaker_spikes_[use_nestml="+str(use_nestml)+"].png")

## Acknowledgements

The authors would like to thank Andrew Rowley for his kind and helpful discussion and feedback.

## Copyright

This file is part of NEST.

Copyright (C) 2004 The NEST Initiative

NEST is free software: you can redistribute it and/or modify it under the terms of the GNU General Public License as published by the Free Software Foundation, either version 2 of the License, or (at your option) any later version.

NEST is distributed in the hope that it will be useful, but WITHOUT ANY WARRANTY; without even the implied warranty of MERCHANTABILITY or FITNESS FOR A PARTICULAR PURPOSE. See the GNU General Public License for more details.

You should have received a copy of the GNU General Public License along with NEST. If not, see http://www.gnu.org/licenses/.